# QAGTA walkthrough

Quantum encoding to adaptive graph construction to attention propagation, one stage at a
time. Install first: `pip install -e ".[dev,viz]"`


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from qagta import PipelineConfig, QuantumAdaptiveGraphPipeline
from qagta.data import generate_multivariate_series, split_normal_anomaly
from qagta.quantum import pairwise_fidelity
from qagta.training.evaluate import comparison_table, evaluate_embeddings

torch.manual_seed(0)

## 1. Data

One-class setup: fit on normal samples only, test on a mix.

In [ ]:
df = generate_multivariate_series(n_samples=300, n_features=10, anomaly_fraction=0.25, seed=42)
split = split_normal_anomaly(df.drop(columns=["attack"]).to_numpy(), df["attack"].to_numpy())
print(f"train {split.x_train.shape} (all normal)")
print(f"test  {split.x_test.shape}, {int(split.y_test.sum())} anomalous")

## 2. Train

Stage one pre-trains the quantum autoencoder; stage two trains topology construction and
propagation on a contextual reconstruction objective.

In [ ]:
config = PipelineConfig()
pipeline = QuantumAdaptiveGraphPipeline(config, input_dim=split.n_features)
pipeline.fit(split.x_train)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(pipeline.history["encoder_loss"]); axes[0].set_title("stage 1: quantum autoencoder")
axes[1].plot(pipeline.history["graph_loss"]);   axes[1].set_title("stage 2: adaptive graph")
for ax in axes: ax.set_xlabel("epoch"); ax.set_ylabel("loss")
fig.tight_layout()

## 3. Inspect the quantum stage

Each qubit must receive a genuinely varying rotation. If the angle spread collapses, all
prepared states look alike, every fidelity approaches 1, and the quantum similarity term
stops carrying information.

In [ ]:
x_test = torch.as_tensor(split.x_test, dtype=torch.float32)
pipeline._eval_mode()
with torch.no_grad():
    angles = pipeline.encoder.encode_angles(x_test)
    latent, states = pipeline._encode(x_test)

print("angle std per qubit:", angles.std(0).numpy().round(3))
print("latent shape:", tuple(latent.shape))

fid = pairwise_fidelity(states)
off = fid[~torch.eye(len(fid), dtype=torch.bool)]
print(f"pairwise fidelity: mean={off.mean():.3f}, std={off.std():.3f}")

plt.figure(figsize=(5, 4))
plt.imshow(fid.numpy(), cmap="viridis"); plt.colorbar(label=r"$|\langle\psi_i|\psi_j\rangle|^2$")
plt.title("quantum fidelity between latent states");

## 4. The learned topology

Edges are recomputed from the current latent states on every pass, so the graph evolves
with the data rather than being fixed up front.

In [ ]:
with torch.no_grad():
    graph = pipeline.constructor(latent, states)
    mixing = pipeline.constructor.edge_learner.mixing

n = latent.shape[0]
print(f"{n} nodes, {graph.edge_index.shape[1]} edges "
      f"(avg degree {graph.edge_index.shape[1]/n:.1f})")
for name, w in zip(["cosine", "learnable", "attention", "fidelity"], mixing):
    print(f"  {name:<10} {float(w):.3f}")

In [ ]:
with torch.no_grad():
    adjacency = pipeline.constructor.edge_learner.dense_adjacency(latent, states)

order = np.argsort(split.y_test)  # group normal / anomalous for readability
plt.figure(figsize=(5.5, 4.5))
plt.imshow(adjacency.numpy()[order][:, order], cmap="magma")
plt.colorbar(label="edge weight"); plt.title("learned adjacency (sorted by class)");

## 5. Compare against ablations

In [ ]:
results = [
    evaluate_embeddings(pipeline.ablation_embed(split.x_train),
                        pipeline.ablation_embed(split.x_test),
                        split.y_test, name="quantum latents only"),
    pipeline.evaluate(split.x_test, split.y_test, name="adaptive graph + GAT"),
]
print(comparison_table(results))

On this synthetic data the graph stage does not beat the latent-only baseline: the
generator produces *point* anomalies that are already separable in latent space, and
neighbourhood propagation can only blur that. Graph structure earns its keep when anomalies
are *relational* — a node whose connections are inconsistent with its neighbourhood.
Benchmark on domain data before drawing conclusions.

## 6. Hybrid optimisation

Quantum weights can co-adapt with the classical stages through the parameter-shift rule,
which stays valid on shot-based backends and real hardware.

In [ ]:
from qagta.training.parameter_shift import expectation_jacobian

circuit = pipeline.encoder.circuit
angles_small = pipeline.encoder.encode_angles(x_test[:8]).detach()

circuit.zero_grad()
circuit(angles_small).sum().backward()
autograd_grad = circuit.weights.grad.clone()
shift_grad = expectation_jacobian(circuit, angles_small).sum(dim=(0, 1))

print("max |autograd - parameter shift|:", float((autograd_grad - shift_grad).abs().max()))